# Limpieza y validación inicial de datos

El objetivo de este notebook es realizar la carga de datos, la validación inicial y la preparación de la estructura base para el análisis descriptivo de osteoporosis.

In [1]:
# Sistema
from pathlib import Path
import warnings

# Manipulación de datos
import numpy as np
import pandas as pd

# Estadística
from scipy import stats
import statsmodels.api as sm

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
RAW_DATA_PATH = Path(
    "/home/marcos-maravilla/análisis_estadístico_osteoporosis/data/raw/BD_Analysis_Osteoporosis.xlsx"
)

if RAW_DATA_PATH.exists():
    print(f"Archivo encontrado correctamente: {RAW_DATA_PATH}")
else:
    print(f"Error: ruta no encontrada -> {RAW_DATA_PATH}")

Archivo encontrado correctamente: /home/marcos-maravilla/análisis_estadístico_osteoporosis/data/raw/BD_Analysis_Osteoporosis.xlsx


In [3]:
try:
    df_raw = pd.read_excel(RAW_DATA_PATH, engine="openpyxl")
    n_rows, n_columns = df_raw.shape
    print(f"Dataset cargado correctamente: {n_rows:,} filas y {n_columns:,} columnas")
    print("\nInformación general del dataset:")
    df_raw.info()
except Exception as error:
    print(f"Error al cargar el archivo Excel: {error}")

Dataset cargado correctamente: 405 filas y 74 columnas

Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 405 entries, 0 to 404
Data columns (total 74 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   1. Sexo                         405 non-null    int64  
 1   2. ECivil                       405 non-null    int64  
 2   3. Edad                         405 non-null    int64  
 3   4. Edad_cat                     405 non-null    int64  
 4   5. Lnac (Entidad)               405 non-null    str    
 5   6. Lnac (Municipio)             405 non-null    str    
 6   7. Lresid (Entidad):            405 non-null    str    
 7   8. Lresid (Municipio):          405 non-null    int64  
 8   9. Escolaridad                  405 non-null    int64  
 9   10. Trabaja                     405 non-null    int64  
 10  11. Tipo_empleo                 405 non-null    int64  
 11  12. Act_empleo     

In [4]:
df_raw.head()

,1. Sexo,2. ECivil,3. Edad,4. Edad_cat,5. Lnac (Entidad),6. Lnac (Municipio),7. Lresid (Entidad):,8. Lresid (Municipio):,9. Escolaridad,10. Trabaja,11. Tipo_empleo,12. Act_empleo,13. Ing_mensual,14. Ing_mensual_cat,15. Derechohabiente,16. Derechohabiente_Cat,17. Salud_Cobertura,18. Salud_Cobertura_cat,19. Enfermedad,20. Enfermedad(cod),21. Enf_cat,22. Tiempo Dx,23. Vive_Con,24. Vive_Con_cat,25. Ayuda_Enf,26. Ayuda_Enf_cat,27. Disp_Aprendizaje,28. Disp_Capacitación,29. Motivo_No_Conocer,30. Grupo_Externo_Enf,31. Tipo_Grupo_Apoyo,32. Realiza_AF,33. Tipo_AF,34. Frecuencia_AF,35. Tiempo_AF,36. Espacio_AF,37. Plan_Alimenticio,38. Autor_Dieta,39. Factores_favorables_dieta,40. Barreras_dieta,Frecuencia_frutas,Frecuencia_verduras,Frecuencia_leguminosas,Frecuencia_cereales_sg,Frecuencia_cereales_cg,Frecuencia_carnes_blancas,Frecuencia_carnes_rojas,Frecuencia_embutidos,Frecuencia_lacteos,Frecuencia_azucares_sg,Frecuencia_azucares_cg,Frecuencia_grasas,Frecuencia_alcohol,Frec_productos_light,Frutas_mas_consumidas,Verduras_mas_consumidas,Leguminosas_mas_consumidas,cereales_sg_frecuentes,cereales_cg_frecuentes,Carnes_blancas_frecuentes,Carnes_rojas_frecuentes,Embutidos_frecuentes,Lacteos_frecuentes,Azucares_sg_frecuentes,Azucares_cg_frecuentes,Grasas_frecuentes,Bebidas_alcoholicas_frecuentes,Productos_light_frecuentes,Dx_Óseo,Dx_Óseo_cat,Peso (kg),Altura (cm),IMC,IMC_Cat
0,1,1,53,1,Jalisco,Cuquio,Jalisco,0,1,0,0,0,"5,000 a 7,499",1,SSA/INSABI,1,SSA/INSABI,2,"Diabetes, Hipertensión arterial, Obesidad, Col...",1,1,4,"Pareja, Hijos",1,Algún familiar,1,1,0,4,0,0,0,0,0,0,0,0,0,No,Ninguno,2,2,4,4,2,2,2,0,2,2,0,3,0,0,"Manzana, Platano","Lechuga, Calabaza, Chayote, Zanahoria",Frijoles,Tortilla de maíz,Ninguno,Pollo,Carne de cerdo,Ninguno,Queso,Ninguna,Ninguno,Aceites,Ninguna,Ninguno,Normal,0,92,160,35.937500,3
1,0,1,65,2,Jalisco,Guadalajara,Jalisco,1,1,1,2,3,"5,000 a 7,499",1,SSA/INSABI,1,SSA/INSABI,2,"Diabetes, Hipertensión arterial, Colesterol el...",1,1,4,Pareja,1,Nadie,0,2,2,3,0,0,0,0,0,0,1,0,0,No,Ninguno,4,2,4,4,1,1,1,0,2,3,0,4,0,0,"Manzana, Mango","Calabaza, Papa, Chayote, Zanahoria","Frijoles, Lentejas","Tortilla de maíz, Arroz","Pasteles, Galletas",Pollo,"Carne de cerdo, Carne de res",Ninguno,"Leche, Queso",Azucar de mesa,Ninguno,"Aderezos, Manteca",Ninguna,Ninguno,Osteopenia,1,69,164,25.654372,2
2,1,1,54,1,Veracruz de Ignacio de la Llave,Veracruz,Veracruz de Ignacio de la Llave,0,1,0,0,0,"0 a 4,999",1,SSA/INSABI,1,SSA/INSABI,2,Sano,8,0,5,Nadie,0,Nadie,0,2,2,3,0,0,0,0,0,0,0,0,0,No,Ninguno,2,3,4,4,0,2,1,0,1,1,0,4,0,0,"Manzana, Platano, Papaya","Calabaza, Chayote, Zanahoria","Frijoles, Lentejas","Tortilla de maíz, Arroz",Ninguno,"Pollo, Pescado","Carne de cerdo, Carne de res",Ninguno,"Leche, Queso",Azucar de mesa,Ninguno,"Aceites, Mantequilla, Manteca",Ninguna,Ninguno,Osteoporósis,1,57,154,24.034407,1
3,1,0,50,1,Jalisco,Tonalá,Jalisco,0,1,1,2,1,"0 a 4,999",1,SSA/INSABI,1,SSA/INSABI,2,"Diabetes, Hipertensión arterial, Obesidad, Col...",1,1,4,Padre o madre,1,Nadie,0,2,2,3,0,0,0,0,0,0,0,0,0,No,Ninguno,0,1,4,1,0,1,3,0,0,0,0,4,0,0,Ninguna,Zanahoria,Frijoles,Tortilla de maíz,Ninguno,Pollo,Carne de res,Ninguno,Ninguno,Ninguna,Ninguno,Manteca,Ninguna,Ninguno,Normal,0,82,154,34.575814,3
4,1,1,70,3,Jalisco,Guadalajara,Jalisco,0,1,0,0,0,"0 a 4,999",1,IMSS,1,IMSS,1,Hipertensión arterial,2,1,4,"Pareja, Hijos",1,Nadie,0,2,2,3,0,0,1,1,2,2,1,0,0,No,Ninguno,2,2,1,1,1,2,2,0,4,0,0,1,0,0,"Manzana, Platano","Calabaza, Chayote, Brocoli, espinaca.",Lentejas,Tortilla de maíz,Ninguno,Pollo,Carne de res,Ninguno,"Leche, Queso",Sustituto de azúcar,Ninguno,Aceite olivo,Ninguna,Ninguno,Osteopenia,1,79,156,32.462196,3


## 2. Estandarización de Variables y Tipos de Datos

Las columnas se renombrarán a formato `snake_case` siguiendo las definiciones del `data_dictionary.md`; además, se ajustarán los tipos de datos entre variables numéricas y categóricas, y se creará la variable dependiente `alteracion_osea` para el análisis descriptivo de osteoporosis.

In [ ]:
column_rename_dict = {
    "1. Sexo": "sexo",
    "2. ECivil": "estado_civil",
    "3. Edad": "edad",
    "4. Edad_cat": "edad_cat",
    "5. Lnac (Entidad)": "lugar_nacimiento_entidad",
    "5. L nac (Entidad)": "lugar_nacimiento_entidad",
    "6. Lnac (Municipio)": "lugar_nacimiento_municipio",
    "6. L nac (Municipio)": "lugar_nacimiento_municipio",
    "7. Lresid (Entidad):": "lugar_residencia_entidad",
    "7. L resid (Entidad)": "lugar_residencia_entidad",
    "8. Lresid (Municipio):": "lugar_residencia_municipio",
    "8. L resid (Municipio)": "lugar_residencia_municipio",
    "9. Escolaridad": "escolaridad",
    "10. Trabaja": "trabaja",
    "11. Tipo_empleo": "tipo_empleo",
    "12. Act_empleo": "act_empleo",
    "13. Ing_mensual": "ing_mensual",
    "14. Ing_mensual_cat": "ing_mensual_cat",
    "15. Derechohabiente": "derechohabiente",
    "16. Derechohabiente_Cat": "derechohabiente_cat",
    "16. Derechohabiente_cat": "derechohabiente_cat",
    "17. Salud_Cobertura": "salud_cobertura",
    "18. Salud_Cobertura_cat": "salud_cobertura_cat",
    "19. Enfermedad": "enfermedad",
    "20. Enfermedad(cod)": "enfermedad_cod",
    "21. Enf_cat": "enfermedad_cat",
    "21. Enfermedad_cat": "enfermedad_cat",
    "22. Tiempo Dx": "tiempo_dx",
    "23. Vive_Con": "vive_con",
    "24. Vive_Con_cat": "vive_con_cat",
    "25. Ayuda_Enf": "ayuda_enf",
    "26. Ayuda_Enf_cat": "ayuda_enf_cat",
    "26. Ayuda_Enf(cod)": "ayuda_enf_cat",
    "27. Disp_Aprendizaje": "disp_aprendizaje",
    "28. Disp_Capacitación": "disp_capacitacion",
    "29. Motivo_No_Conocer": "motivo_no_conocer",
    "30. Grupo_Externo_Enf": "grupo_externo_enf",
    "31. Tipo_Grupo_Apoyo": "tipo_grupo_apoyo",
    "32. Realiza_AF": "realiza_af",
    "33. Tipo_AF": "tipo_af",
    "34. Frecuencia_AF": "frecuencia_af",
    "35. Tiempo_AF": "tiempo_af",
    "36. Espacio_AF": "espacio_af",
    "37. Plan_Alimenticio": "plan_alimenticio",
    "38. Autor_Dieta": "autor_dieta",
    "39. Factores_favorables_dieta": "factores_favorables_dieta",
    "40. Barreras_dieta": "barreras_dieta",
    "41. Dx_Óseo": "dx_oseo",
    "Dx_Óseo": "dx_oseo",
    "42. Dx_Óseo_cat": "dx_oseo_cat",
    "Dx_Óseo_cat": "dx_oseo_cat",
    "43. Peso (kg)": "peso_(kg)",
    "Peso (kg)": "peso_(kg)",
    "44. Altura (cm)": "altura_(cm)",
    "Altura (cm)": "altura_(cm)",
    "45. IMC": "imc",
    "IMC": "imc",
    "46. IMC_Cat": "imc_cat",
    "IMC_Cat": "imc_cat",
}

df_clean = df_raw.rename(columns=column_rename_dict).copy()
df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')

print(f"Columnas originales: {df_raw.shape[1]}")
print(f"Columnas estandarizadas: {df_clean.shape[1]}")
df_clean.columns[:15].tolist()


In [ ]:
continuous_columns = ["edad", "peso_(kg)", "altura_(cm)", "imc"]

for column in continuous_columns:
    if column not in df_clean.columns:
        raise KeyError(f"No se encontró la columna continua esperada: {column}")
    df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

if "dx_oseo_cat" in df_clean.columns:
    dx_oseo_numeric = pd.to_numeric(df_clean["dx_oseo_cat"], errors="coerce")
    df_clean["alteracion_osea"] = np.where(
        dx_oseo_numeric.eq(0),
        0,
        np.where(dx_oseo_numeric.eq(1), 1, np.nan),
    )
elif "dx_oseo" in df_clean.columns:
    dx_oseo_normalized = (
        df_clean["dx_oseo"]
        .astype("string")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
    )
    df_clean["alteracion_osea"] = np.where(
        dx_oseo_normalized.eq("normal"),
        0,
        np.where(dx_oseo_normalized.isin(["osteopenia", "osteoporosis"]), 1, np.nan),
    )
else:
    raise KeyError("No se encontró una columna de diagnóstico óseo para construir alteracion_osea.")

df_clean["alteracion_osea"] = pd.to_numeric(
    df_clean["alteracion_osea"], errors="coerce"
).astype("Int64")

categorical_columns = ["sexo", "alteracion_osea", "trabaja", "realiza_af"]

for column in categorical_columns:
    if column not in df_clean.columns:
        raise KeyError(f"No se encontró la columna categórica esperada: {column}")
    df_clean[column] = df_clean[column].astype("category")

df_clean[continuous_columns + categorical_columns].head()


In [ ]:
print("Distribución de alteracion_osea:")
print(df_clean["alteracion_osea"].value_counts(dropna=False))

df_clean[["edad", "imc", "sexo", "alteracion_osea"]].info()
